In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
from scipy.stats import gaussian_kde

In [109]:
df = pd.read_csv("../results/performance_detailed.csv")
to_drop_cols = [col for col in df.columns if not df[col].any() and df[col].dtype == 'bool'] # all columns that are empty and boolean
df = df.drop(columns=['matrix_size'] + to_drop_cols)
df

,file,reduced_size,mhs_count,computation_time,hypotheses_generated,levels_explored,max_level_size,file_size_mb,ones_count
0,74181.000.matrix,3×33,271,3.867848,2867,3,2356,0.001053,54
1,74181.001.matrix,2×24,95,0.028332,260,2,236,0.000925,29
2,74181.002.matrix,2×24,23,0.018339,187,2,163,0.000925,34
3,74181.003.matrix,2×10,1,0.002642,55,2,45,0.000924,11
4,74181.004.matrix,2×15,14,0.004027,59,2,44,0.000925,25
5,74181.005.matrix,4×26,42,0.218860,1227,4,519,0.001181,68
6,74181.006.matrix,2×17,29,0.021456,147,2,130,0.000924,18
7,74181.007.matrix,2×5,5,0.002753,15,2,10,0.000924,6
8,74181.008.matrix,3×19,1,0.784333,1006,3,816,0.001052,23
9,74181.009.matrix,3×23,68,0.152109,730,3,480,0.001053,41


In [110]:
def parse_reduced_size(s):
    try:
        r, c = map(int, s.split('×'))
        return pd.Series({'rows': r, 'cols': c, 'size': r * c})
    except:
        return pd.Series({'rows': 0, 'cols': 0, 'size': 0})

df = pd.concat([df, df['reduced_size'].apply(parse_reduced_size)], axis=1)
df = df.drop(columns=['reduced_size'])
df

,file,mhs_count,computation_time,hypotheses_generated,levels_explored,max_level_size,file_size_mb,ones_count,rows,cols,size
0,74181.000.matrix,271,3.867848,2867,3,2356,0.001053,54,3,33,99
1,74181.001.matrix,95,0.028332,260,2,236,0.000925,29,2,24,48
2,74181.002.matrix,23,0.018339,187,2,163,0.000925,34,2,24,48
3,74181.003.matrix,1,0.002642,55,2,45,0.000924,11,2,10,20
4,74181.004.matrix,14,0.004027,59,2,44,0.000925,25,2,15,30
5,74181.005.matrix,42,0.218860,1227,4,519,0.001181,68,4,26,104
6,74181.006.matrix,29,0.021456,147,2,130,0.000924,18,2,17,34
7,74181.007.matrix,5,0.002753,15,2,10,0.000924,6,2,5,10
8,74181.008.matrix,1,0.784333,1006,3,816,0.001052,23,3,19,57
9,74181.009.matrix,68,0.152109,730,3,480,0.001053,41,3,23,69


In [111]:
df_numeric = df.copy()
for col in df_numeric.columns:
    original_dtype = df_numeric[col].dtype
    df_numeric[col] = pd.to_numeric(df_numeric[col], errors='coerce')
    if df_numeric[col].isnull().all() and not pd.api.types.is_numeric_dtype(original_dtype):
        print(f"Avviso: La colonna '{col}' è diventata tutta NaN dopo la conversione. Probabilmente non era numerica.")

initial_cols = set(df_numeric.columns)
df_numeric.dropna(axis=1, how='all', inplace=True)
dropped_cols_all_nan = list(initial_cols - set(df_numeric.columns))
if dropped_cols_all_nan:
    print(f"\nColonne rimosse perché interamente NaN dopo conversione: {dropped_cols_all_nan}")
else:
    print("\nNessuna colonna rimossa perché interamente NaN dopo conversione.")

df_numeric.info()

cols_to_plot = [col for col in df_numeric.columns if pd.api.types.is_numeric_dtype(df_numeric[col])]

if not cols_to_plot:
    print("\nErrore: Nessuna colonna numerica valida trovata per il plotting.")
    exit()

for col in cols_to_plot:
    plot_data = df_numeric[[col]].dropna()

    try:
        mean_val = plot_data[col].mean()
        std_val = plot_data[col].std()
        median_val = plot_data[col].median()
        min_val = plot_data[col].min()
        max_val = plot_data[col].max()

        stats_text = '\n'.join((
            r'µ=%.2f' % mean_val,
            r'σ=%.2f' % std_val,
            r'median=%.2f' % median_val,
            r'min=%.2f' % min_val,
            r'max=%.2f' % max_val
        ))

        fig = px.histogram(
            plot_data,
            x=col,
            histnorm='density',
            # marginal="kde", #! verificare perchè non funziona
            color_discrete_sequence=['red'],
            opacity=0.75,
            title=f'{col}',
            labels={col: col.replace("_", " ").title(), 'density': 'Density'},
            width=900, height=500,
            nbins=20, 
            marginal="violin"
        )
        
        x_range_kde = np.linspace(min_val - (max_val - min_val) * 0.2, max_val + (max_val - min_val) * 0.2, 500)
        kde = gaussian_kde(plot_data[col])
        kde_y = kde(x_range_kde)

        fig.add_trace(go.Scatter(
            x=x_range_kde,
            y=kde_y,
            mode='lines',
            line=dict(color='lightcoral', width=5),
            name='KDE'
        ))

        fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='lightgrey')
        fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='lightgrey')

        fig.add_annotation(
        xref="paper",
        yref="paper",
        x=1.05,
        y=0.5,
        text=stats_text,
        showarrow=False,
        font=dict(
            size=12,
            color="black"
        ),
        align="left",
        xanchor='left',
        yanchor='middle',
        bgcolor="lightgrey",
        bordercolor="black",
        borderwidth=1,
        borderpad=5
        )

        fig.update_layout(
            margin=dict(
                l=80,  
                r=350, 
                b=80, 
                t=80   
            ),
        )

        fig.show()

    except Exception as e:
        print(f"  ERRORE CRITICO: Non è stato possibile generare il grafico per '{col}'. Errore: {e}")
        print(f"  Dati problematici per '{col}':")
        print(plot_data[col].describe())
        print(plot_data[col].head())
        print(f"  Conteggio NaN: {plot_data[col].isnull().sum()}")

Avviso: La colonna 'file' è diventata tutta NaN dopo la conversione. Probabilmente non era numerica.

Colonne rimosse perché interamente NaN dopo conversione: ['file']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 10 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   mhs_count             10 non-null     int64  
 1   computation_time      10 non-null     float64
 2   hypotheses_generated  10 non-null     int64  
 3   levels_explored       10 non-null     int64  
 4   max_level_size        10 non-null     int64  
 5   file_size_mb          10 non-null     float64
 6   ones_count            10 non-null     int64  
 7   rows                  10 non-null     int64  
 8   cols                  10 non-null     int64  
 9   size                  10 non-null     int64  
dtypes: float64(2), int64(8)
memory usage: 932.0 bytes


In [112]:
fig = px.scatter(
    df,                 
    x='size',                      
    y='computation_time',                  
    color='mhs_count',
    size='mhs_count',                 
    hover_data=['size', 'computation_time', 'mhs_count'],
    title='Scatter Plot Interattivo: computation_time vs size',
    labels={'size': 'Elementi totali della matrice', 'computation_time': 'Tempo computazionale (s)', 'mhs_count': 'Numero di MHS'}
    # marginal_x="box"
)

fig.show()

In [113]:
fig = px.scatter(
    df,
    x='size',
    y='computation_time',
    color='ones_count',  # Colora i punti in base al numero di '1'
    size='ones_count', # La dimensione del punto indica il numero di '1' nella matrice
    hover_data=['mhs_count', 'hypotheses_generated', 'levels_explored', 'file_size_mb', 'ones_count', 'cols'],
    title='Tempo di Calcolo vs. Dimensione della Matrice (Colorato per 1)',
    labels={'size': 'Dimensione Matrice (righe*colonne)', 'computation_time': 'Tempo di Calcolo (s)'},
    log_y=True # Utile se il tempo di calcolo cresce esponenzialmente
)
fig.show()

In [114]:
fig = px.scatter(
    df,
    x='hypotheses_generated',
    y='computation_time',
    color='size', # Colora in base alla dimensione del problema
    size='mhs_count', # La dimensione del punto indica il numero di MHS trovati
    hover_data=['mhs_count', 'levels_explored', 'max_level_size', 'rows', 'cols', 'file_size_mb'],
    title='Tempo di Calcolo vs. Ipotesi Generate (Colorato per Dimensione Matrice)',
    labels={'hypotheses_generated': 'Ipotesi Generate', 'computation_time': 'Tempo di Calcolo (s)'},
    log_x=True, # Utile se le ipotesi generate crescono molto
    log_y=True
)
fig.show()

In [115]:
fig = px.box(
    df,
    x='rows',
    y='computation_time',
    color='cols', # Mostra box plot separati per 'rows', colorati per 'cols'
    points="all", # Mostra tutti i punti dati oltre al box plot
    hover_data=['mhs_count', 'size', 'file_size_mb', 'ones_count'],
    title='Distribuzione del Tempo di Calcolo per Numero di Righe e Colonne',
    labels={'rows': 'Numero di Righe', 'computation_time': 'Tempo di Calcolo (s)'},
    log_y=True
)
fig.show()

In [116]:
fig = px.scatter(
    df,
    x='size',
    y='computation_time',
    color='ones_count', # Colora in base al numero di '1'
    facet_col='cols', # Crea colonne separate per ogni valore di 'cols'
    facet_row='rows', # Crea righe separate per ogni valore di 'rows'
    hover_data=['mhs_count', 'hypotheses_generated', 'levels_explored', 'file_size_mb'],
    title='Tempo di Calcolo vs. Dimensione Matrice per Righe e Colonne',
    labels={'size': 'Dimensione Matrice', 'computation_time': 'Tempo di Calcolo (s)'},
    log_y=True,
    height=800 # Aumenta l'altezza per una migliore visualizzazione dei facet
)
fig.show()

In [117]:
df_sorted = df.sort_values(by=['rows', 'cols', 'size'])

fig = px.line(
    df_sorted,
    x='size',                  # Variabile sull'asse X (dimensione della matrice)
    y='computation_time',      # Variabile sull'asse Y (tempo di calcolo)
    color='rows',              # Crea una linea separata e colorata per ogni valore di 'rows'
    line_dash='cols',          # (Opzionale) Aggiunge uno stile di linea diverso per ogni valore di 'cols'
    hover_data=['mhs_count', 'hypotheses_generated', 'levels_explored', 'file_size_mb', 'ones_count', 'rows', 'cols'],
    title='Tempo di Calcolo vs. Dimensione Matrice per Diverse Config. (rows/cols)',
    labels={'size': 'Dimensione Matrice (righe*colonne)', 'computation_time': 'Tempo di Calcolo (s)',
            'rows': 'Num. Righe', 'cols': 'Num. Colonne'},
    log_y=True,                # Utile se il tempo di calcolo cresce esponenzialmente
    markers=True               # Mostra un marcatore per ogni punto dato
)

fig.update_traces(mode='lines+markers') # Assicurati che vengano mostrati sia linee che marcatori

fig.show()

In [118]:
fig = px.line(
    df.sort_values('cols'),
    x='cols',
    y='computation_time',
    line_dash='rows',
    title='Computation Time vs. Number of Columns',
    labels={'cols': 'Number of Columns', 'computation_time': 'Computation Time (s)'},
    markers=True
)

fig.show()

In [119]:
fig = px.line(
    df.sort_values('ones_count'),
    x='ones_count',
    y='computation_time',
    title='Computation Time vs. Number of Ones',
    labels={'ones_count': 'Number of Ones', 'computation_time': 'Computation Time (s)'},
    markers=True
)

fig.show()

In [120]:
fig = px.line(
    df.sort_values('cols'),
    x='cols',
    y='computation_time',
    title='Computation Time vs. Number of Columns',
    labels={'cols': 'Number of Columns', 'computation_time': 'Computation Time (s)'},
    markers=True
)

fig.show()

# df.sort_values('cols').plot(x='cols', y='computation_time', marker='o', legend=False)
# plt.xlabel('Numero colonne')
# plt.ylabel('Tempo di calcolo (s)')
# plt.title('Tempo vs Numero colonne')
# plt.grid(True)
# plt.show()

In [121]:
fig = px.line(
    df.sort_values('rows'),
    x='rows',
    y='computation_time',
    title='Computation Time vs. Number of Rows',
    labels={'rows': 'Number of Rows', 'computation_time': 'Computation Time (s)'},
    markers=True
)

fig.show()

# df.sort_values('rows').plot(x='rows', y='computation_time', marker='o', legend=False)
# plt.xlabel('Numero righe')
# plt.ylabel('Tempo di calcolo (s)')
# plt.title('Tempo vs Numero righe')
# plt.grid(True)
# plt.show()

In [122]:
fig = px.line(
    df.sort_values('size'),
    x='size',
    y='computation_time',
    title='Computation Time vs. Number of Elements',
    labels={'size': 'Number of Elements', 'computation_time': 'Computation Time (s)'},
    markers=True
)

fig.show()

In [123]:
fig = px.line(
    df.sort_values('mhs_count'),
    x='mhs_count',
    y='computation_time',
    title='Computation Time vs. Number of MHS found',
    labels={'mhs_count': 'Number of MHS found', 'computation_time': 'Computation Time (s)'},
    markers=True
)

fig.show()

In [124]:
fig = px.line(
    df.sort_values('mhs_count'),
    x='mhs_count',
    y='size',
    title='Size vs. Number of MHS found',
    labels={'mhs_count': 'Number of MHS found', 'size': 'Size'},
    markers=True
)

fig.show()

In [125]:
fig = px.line(
    df.sort_values('file_size_mb'),
    x='file_size_mb',
    y='computation_time',
    title='Computation Time vs. File Size',
    labels={'file_size_mb': 'File Size (MB)', 'computation_time': 'Computation Time (s)'},
    markers=True
)

fig.show()

In [126]:
fig = px.line(
    df.sort_values('levels_explored'),
    x='levels_explored',
    y='computation_time',
    title='Computation Time vs. Levels Explored',
    labels={'levels_explored': 'Levels Explored', 'computation_time': 'Computation Time (s)'},
    markers=True
)

fig.show()

In [127]:
fig = px.line(
    df.sort_values('max_level_size'),
    x='max_level_size',
    y='computation_time',
    title='Computation Time vs. Max Level Size',
    labels={'max_level_size': 'Max Level Size', 'computation_time': 'Computation Time (s)'},
    markers=True
)

fig.show()